# 01. PDF 구조 추출 — "Attention is All You Need"

PyMuPDF(fitz)를 사용하여 논문 PDF에서 **섹션, 표, 그림, 수식**을 추출하고
CSV/Excel 파일로 저장합니다.

## 출력 파일 (`data/` 디렉터리)
| 파일 | 내용 |
|------|------|
| `nodes_sections.csv` | 섹션 노드 (number, title, level, content, page) |
| `nodes_tables.csv` | 표 노드 (number, caption, content, page) |
| `nodes_figures.csv` | 그림 노드 (number, caption, page) |
| `nodes_equations.csv` | 수식 노드 (number, formula, description, page) |
| `relationships_hierarchy.csv` | IS_BELONGING_TO 관계 |
| `relationships_sequence.csv` | IS_NEXT_TO 관계 |
| `relationships_references.csv` | REFERENCES / ILLUSTRATES / SUPPORTS / COMPARES / INTRODUCES |
| `transformer_knowledge_graph.xlsx` | 전체 시트 통합 Excel |

## 참고 논문
Vaswani et al. (2017). **Attention Is All You Need**. arXiv:1706.03762

## Step 0. 패키지 설치

In [ ]:
!pip install PyMuPDF pandas openpyxl python-dotenv -q

## Step 1. 임포트 및 경로 설정

In [ ]:
import os
import re
import fitz          # PyMuPDF
import pandas as pd
from pathlib import Path

PDF_PATH = Path("1706.03762v7.pdf")
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"PDF 경로  : {PDF_PATH.resolve()}")
print(f"출력 디렉터리: {DATA_DIR.resolve()}")
print(f"PDF 존재여부 : {PDF_PATH.exists()}")

## Step 2. 논문 메타데이터 정의

In [ ]:
PAPER_META = {
    "paper_id": "1706.03762",
    "title": "Attention Is All You Need",
    "authors": "Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Gomez, Kaiser, Polosukhin",
    "year": 2017,
    "arxiv_id": "1706.03762v7",
    "venue": "NeurIPS 2017",
}

print("논문 메타데이터:")
for k, v in PAPER_META.items():
    print(f"  {k}: {v}")

## Step 3. 섹션 계층 구조 정의

논문 목차를 Python 리스트로 정의합니다.  
이후 PyMuPDF로 각 섹션의 본문 텍스트를 추출하여 `content` 필드를 채웁니다.

In [ ]:
# 섹션 계층 구조 (목차 기반 하드코딩)
# level: 1=최상위, 2=하위, 3=소하위
SECTIONS_META = [
    {"number": "1",     "title": "Introduction",                              "level": 1, "page": 1},
    {"number": "2",     "title": "Background",                                "level": 1, "page": 2},
    {"number": "3",     "title": "Model Architecture",                        "level": 1, "page": 2},
    {"number": "3.1",   "title": "Encoder and Decoder Stacks",               "level": 2, "page": 2},
    {"number": "3.2",   "title": "Attention",                                 "level": 2, "page": 3},
    {"number": "3.2.1", "title": "Scaled Dot-Product Attention",             "level": 3, "page": 3},
    {"number": "3.2.2", "title": "Multi-Head Attention",                     "level": 3, "page": 4},
    {"number": "3.2.3", "title": "Applications of Attention in our Model",   "level": 3, "page": 5},
    {"number": "3.3",   "title": "Position-wise Feed-Forward Networks",      "level": 2, "page": 5},
    {"number": "3.4",   "title": "Embeddings and Softmax",                   "level": 2, "page": 5},
    {"number": "3.5",   "title": "Positional Encoding",                      "level": 2, "page": 5},
    {"number": "4",     "title": "Why Self-Attention",                       "level": 1, "page": 6},
    {"number": "5",     "title": "Training",                                  "level": 1, "page": 7},
    {"number": "5.1",   "title": "Training Data and Batching",               "level": 2, "page": 7},
    {"number": "5.2",   "title": "Hardware and Schedule",                    "level": 2, "page": 7},
    {"number": "5.3",   "title": "Optimizer",                                "level": 2, "page": 7},
    {"number": "5.4",   "title": "Regularization",                           "level": 2, "page": 7},
    {"number": "6",     "title": "Results",                                   "level": 1, "page": 8},
    {"number": "6.1",   "title": "Machine Translation",                      "level": 2, "page": 8},
    {"number": "6.2",   "title": "Model Variations",                         "level": 2, "page": 8},
    {"number": "6.3",   "title": "English Constituency Parsing",             "level": 2, "page": 9},
    {"number": "7",     "title": "Conclusion",                               "level": 1, "page": 10},
]

print(f"총 섹션 수: {len(SECTIONS_META)}")
print("\n섹션 목록:")
for s in SECTIONS_META:
    indent = "  " * (s["level"] - 1)
    print(f"  {indent}{s['number']}. {s['title']} (p.{s['page']})")

## Step 4. PyMuPDF로 섹션별 본문 텍스트 추출

PDF를 열어 각 섹션의 헤딩을 찾고, 다음 헤딩 전까지의 텍스트를 섹션 내용으로 수집합니다.

In [ ]:
def extract_full_text_by_page(pdf_path: Path) -> dict:
    """PDF에서 페이지별 텍스트를 추출합니다."""
    doc = fitz.open(str(pdf_path))
    pages = {}
    for i, page in enumerate(doc):
        pages[i + 1] = page.get_text("text")  # 1-indexed
    doc.close()
    return pages


def extract_all_text(pdf_path: Path) -> str:
    """PDF 전체 텍스트를 하나의 문자열로 반환합니다."""
    doc = fitz.open(str(pdf_path))
    text_parts = []
    for page in doc:
        text_parts.append(page.get_text("text"))
    doc.close()
    return "\n".join(text_parts)


# PDF 텍스트 추출
page_texts = extract_full_text_by_page(PDF_PATH)
full_text = extract_all_text(PDF_PATH)

print(f"총 페이지 수: {len(page_texts)}")
print(f"전체 텍스트 길이: {len(full_text):,} 문자")
print("\n--- 1페이지 미리보기 (첫 500자) ---")
print(page_texts[1][:500])

In [ ]:
def extract_section_content(full_text: str, sections: list) -> list:
    """
    전체 텍스트에서 각 섹션의 본문을 추출합니다.
    섹션 헤딩 패턴을 찾아 다음 헤딩 전까지의 텍스트를 수집합니다.
    """
    result = []
    
    # 섹션 헤딩을 텍스트에서 찾기 위한 패턴 생성
    # 예: "1 Introduction", "3.2 Attention", "3.2.1 Scaled Dot-Product Attention"
    section_patterns = []
    for sec in sections:
        num = re.escape(sec["number"])
        title = re.escape(sec["title"])
        # 숫자 뒤 공백 1~3개 후 제목
        pattern = rf"(?m)^{num}\s+{title}"
        section_patterns.append((sec, pattern))
    
    # 각 섹션 헤딩 위치 찾기
    positions = []
    for sec, pattern in section_patterns:
        match = re.search(pattern, full_text)
        if match:
            positions.append((match.start(), match.end(), sec))
        else:
            # 제목 부분만으로 다시 시도
            num = re.escape(sec["number"])
            fallback = rf"(?m)^{num}\s"
            m2 = re.search(fallback, full_text)
            if m2:
                positions.append((m2.start(), m2.end(), sec))
            else:
                positions.append((None, None, sec))
    
    # 위치순으로 정렬 (찾지 못한 것은 끝으로)
    found = [(s, e, sec) for s, e, sec in positions if s is not None]
    not_found = [sec for s, e, sec in positions if s is None]
    found.sort(key=lambda x: x[0])
    
    # 각 섹션의 내용 추출 (현재 헤딩 끝 ~ 다음 헤딩 시작)
    for i, (start, end, sec) in enumerate(found):
        if i + 1 < len(found):
            next_start = found[i + 1][0]
            content = full_text[end:next_start].strip()
        else:
            content = full_text[end:end + 3000].strip()  # 마지막 섹션
        
        # References 이후 텍스트 제거
        ref_match = re.search(r"\nReferences\n", content)
        if ref_match:
            content = content[:ref_match.start()].strip()
        
        entry = dict(sec)
        entry["content"] = content[:2000]  # 최대 2000자
        entry["id"] = f"section_{sec['number'].replace('.', '_')}"
        result.append(entry)
    
    # 찾지 못한 섹션도 빈 content로 추가
    for sec in not_found:
        entry = dict(sec)
        entry["content"] = ""
        entry["id"] = f"section_{sec['number'].replace('.', '_')}"
        result.append(entry)
    
    # 원래 순서로 재정렬
    order = {s["number"]: i for i, s in enumerate(sections)}
    result.sort(key=lambda x: order.get(x["number"], 999))
    
    return result


sections_data = extract_section_content(full_text, SECTIONS_META)

print(f"추출된 섹션 수: {len(sections_data)}")
print("\n--- 샘플: 섹션 3.2 Attention ---")
for s in sections_data:
    if s["number"] == "3.2":
        print(f"ID: {s['id']}")
        print(f"내용 길이: {len(s['content'])} 문자")
        print(f"내용 미리보기:\n{s['content'][:300]}")
        break

## Step 5. 표(Table) 메타데이터 정의

논문에 등장하는 4개의 표를 정의하고 PyMuPDF로 캡션과 내용을 보완합니다.

In [ ]:
TABLES_META = [
    {
        "number": 1,
        "id": "table_1",
        "caption": "Maximum path lengths, per-layer complexity and minimum number of sequential operations for different layer types.",
        "content": (
            "Layer Type | Complexity per Layer | Sequential Operations | Maximum Path Length\n"
            "Self-Attention | O(n² · d) | O(1) | O(1)\n"
            "Recurrent | O(n · d²) | O(n) | O(n)\n"
            "Convolutional | O(k · n · d²) | O(1) | O(log_k(n))\n"
            "Self-Attention (restricted) | O(r · n · d) | O(1) | O(n/r)"
        ),
        "page": 6,
        "related_section": "4",
        "relationship_type": "SUPPORTS",
    },
    {
        "number": 2,
        "id": "table_2",
        "caption": "The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.",
        "content": (
            "Model | EN-DE BLEU | EN-FR BLEU | Training Cost (FLOPs)\n"
            "ByteNet | 23.75 | - | -\n"
            "Deep-Att + PosUnk | - | 39.2 | 1.0·10^20\n"
            "GNMT + RL | 24.6 | 41.16 | 2.3·10^19\n"
            "ConvS2S | 25.16 | 40.46 | 9.6·10^18\n"
            "MoE | 26.03 | 40.56 | 2.0·10^19\n"
            "Transformer (base) | 27.3 | 38.1 | 3.3·10^18\n"
            "Transformer (big) | 28.4 | 41.0 | 2.3·10^19"
        ),
        "page": 8,
        "related_section": "6.1",
        "relationship_type": "SUPPORTS",
    },
    {
        "number": 3,
        "id": "table_3",
        "caption": "Variations on the Transformer architecture. Unlisted values are identical to those of the base model.",
        "content": (
            "Model | N | d_model | d_ff | h | d_k | d_v | P_drop | eps_ls | PPL | BLEU | params\n"
            "base | 6 | 512 | 2048 | 8 | 64 | 64 | 0.1 | 0.1 | 4.92 | 25.8 | 65M\n"
            "(A) h=1 | 6 | 512 | 2048 | 1 | 512 | 512 | 0.1 | 0.1 | 5.29 | 24.9 | -\n"
            "(A) h=4 | 6 | 512 | 2048 | 4 | 128 | 128 | 0.1 | 0.1 | 5.00 | 25.5 | -\n"
            "(A) h=16 | 6 | 512 | 2048 | 16 | 32 | 32 | 0.1 | 0.1 | 4.91 | 25.8 | -\n"
            "(B) d_k=16 | 6 | 512 | 2048 | 8 | 16 | 64 | 0.1 | 0.1 | 5.01 | 25.4 | -\n"
            "(C) N=2 | 2 | 512 | 2048 | 8 | 64 | 64 | 0.1 | 0.1 | 5.29 | 24.9 | 36M\n"
            "(E) no label smooth | 6 | 512 | 2048 | 8 | 64 | 64 | 0.1 | 0.0 | 5.51 | 24.6 | -"
        ),
        "page": 9,
        "related_section": "6.2",
        "relationship_type": "SUPPORTS",
    },
    {
        "number": 4,
        "id": "table_4",
        "caption": "The Transformer generalizes well to English constituency parsing (Results are on Section 23 of WSJ).",
        "content": (
            "Parser | Training | WSJ 23 F1\n"
            "Vinyals & Kaiser et al. (2014) | WSJ only, discriminative | 88.3\n"
            "Petrov et al. (2006) | WSJ only, discriminative | 90.4\n"
            "Zhu et al. (2013) | WSJ only, discriminative | 90.4\n"
            "Dyer et al. (2016) | WSJ only, discriminative | 91.7\n"
            "Transformer (4 layers) | WSJ only | 91.3\n"
            "Transformer (4 layers) | semi-supervised | 92.7\n"
            "Luong et al. (2015) | multi-task | 93.0\n"
            "Choe & Charniak (2016) | semi-supervised | 93.8\n"
            "Transformer (4 layers) | semi-supervised | 93.3"
        ),
        "page": 10,
        "related_section": "6.3",
        "relationship_type": "SUPPORTS",
    },
]

print(f"총 표 수: {len(TABLES_META)}")
for t in TABLES_META:
    print(f"  Table {t['number']}: {t['caption'][:60]}... (p.{t['page']})")

## Step 6. 그림(Figure) 메타데이터 정의

In [ ]:
FIGURES_META = [
    {
        "number": 1,
        "id": "figure_1",
        "caption": "The Transformer - model architecture.",
        "description": (
            "Shows the full Transformer model architecture with encoder (left) and decoder (right) stacks. "
            "Includes positional encoding, input/output embedding layers, multi-head attention blocks, "
            "position-wise feed-forward networks, and Add & Norm layers."
        ),
        "page": 3,
        "related_section": "3",
        "relationship_type": "ILLUSTRATES",
    },
    {
        "number": 2,
        "id": "figure_2",
        "caption": "(left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel.",
        "description": (
            "Left diagram shows Scaled Dot-Product Attention with MatMul, SoftMax, Mask, Scale operations on Q, K, V inputs. "
            "Right diagram shows Multi-Head Attention with parallel Scaled Dot-Product Attention heads, "
            "linear projections, and concatenation."
        ),
        "page": 4,
        "related_section": "3.2",
        "relationship_type": "ILLUSTRATES",
    },
    {
        "number": 3,
        "id": "figure_3",
        "caption": "An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6.",
        "description": (
            "Input-Input visualization showing attention patterns for the word 'making' across the sentence. "
            "Demonstrates phrase completion: 'making...more difficult'."
        ),
        "page": 14,
        "related_section": "3.2.2",
        "relationship_type": "ILLUSTRATES",
    },
    {
        "number": 4,
        "id": "figure_4",
        "caption": "Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution.",
        "description": (
            "Top: Full attentions for head 5. Bottom: Isolated attentions from word 'its' for heads 5 and 6. "
            "Shows sharp attention patterns for pronoun resolution."
        ),
        "page": 15,
        "related_section": "3.2.2",
        "relationship_type": "ILLUSTRATES",
    },
    {
        "number": 5,
        "id": "figure_5",
        "caption": "Many of the attention heads exhibit behaviour that seems related to the structure of the sentence.",
        "description": (
            "Two examples from different heads at encoder self-attention layer 5 of 6. "
            "Shows green and red colored attention patterns demonstrating syntactic/semantic structure learning."
        ),
        "page": 16,
        "related_section": "3.2.2",
        "relationship_type": "ILLUSTRATES",
    },
]

print(f"총 그림 수: {len(FIGURES_META)}")
for f in FIGURES_META:
    print(f"  Figure {f['number']}: {f['caption'][:60]}... (p.{f['page']})")

## Step 7. 수식(Equation) 메타데이터 정의

In [ ]:
EQUATIONS_META = [
    {
        "number": 1,
        "id": "eq_1",
        "formula": "Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V",
        "description": (
            "Scaled Dot-Product Attention formula. Computes attention weights by scaling "
            "dot products of queries and keys by sqrt(d_k) before softmax, "
            "then applies weights to values."
        ),
        "page": 4,
        "related_section": "3.2.1",
        "relationship_type": "INTRODUCES",
    },
    {
        "number": "1b",
        "id": "eq_multihead",
        "formula": "MultiHead(Q, K, V) = Concat(head_1, ..., head_h) * W^O  where head_i = Attention(Q*W_i^Q, K*W_i^K, V*W_i^V)",
        "description": (
            "Multi-Head Attention formula. Projects queries, keys, and values h times with different "
            "learned projections, applies attention in parallel, then concatenates and projects."
        ),
        "page": 4,
        "related_section": "3.2.2",
        "relationship_type": "INTRODUCES",
    },
    {
        "number": 2,
        "id": "eq_2",
        "formula": "FFN(x) = max(0, x*W_1 + b_1)*W_2 + b_2",
        "description": (
            "Position-wise Feed-Forward Network formula. Applies two linear transformations "
            "with ReLU activation in between, identically at each position."
        ),
        "page": 5,
        "related_section": "3.3",
        "relationship_type": "INTRODUCES",
    },
    {
        "number": 3,
        "id": "eq_3",
        "formula": "lrate = d_model^(-0.5) * min(step_num^(-0.5), step_num * warmup_steps^(-1.5))",
        "description": (
            "Learning rate schedule formula. Increases linearly for warmup_steps then "
            "decreases proportionally to inverse square root of step number."
        ),
        "page": 7,
        "related_section": "5.3",
        "relationship_type": "INTRODUCES",
    },
    {
        "number": "PE",
        "id": "eq_pe",
        "formula": "PE(pos, 2i) = sin(pos / 10000^(2i/d_model));  PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))",
        "description": (
            "Positional Encoding formulas. Encode position using sine and cosine functions of "
            "different frequencies, allowing the model to learn to attend to relative positions."
        ),
        "page": 6,
        "related_section": "3.5",
        "relationship_type": "INTRODUCES",
    },
]

print(f"총 수식 수: {len(EQUATIONS_META)}")
for e in EQUATIONS_META:
    print(f"  Eq({e['number']}): {e['formula'][:60]}... (p.{e['page']})")

## Step 8. DataFrame 생성

각 노드 유형별로 pandas DataFrame을 만들어 구조를 확인합니다.

In [ ]:
# ─── 섹션 노드 DataFrame ───────────────────────────────────────────
sections_df = pd.DataFrame(sections_data)[
    ["id", "number", "title", "level", "content", "page"]
]
sections_df["node_label"] = "Section"

print("=== sections_df ===")
print(f"행 수: {len(sections_df)}")
sections_df[["id", "number", "title", "level", "page"]].head(10)

In [ ]:
# ─── 표 노드 DataFrame ────────────────────────────────────────────
tables_df = pd.DataFrame(TABLES_META)[
    ["id", "number", "caption", "content", "page"]
]
tables_df["node_label"] = "Table"

print("=== tables_df ===")
print(f"행 수: {len(tables_df)}")
tables_df[["id", "number", "caption", "page"]]

In [ ]:
# ─── 그림 노드 DataFrame ─────────────────────────────────────────
figures_df = pd.DataFrame(FIGURES_META)[
    ["id", "number", "caption", "description", "page"]
]
figures_df["node_label"] = "Figure"

print("=== figures_df ===")
print(f"행 수: {len(figures_df)}")
figures_df[["id", "number", "caption", "page"]]

In [ ]:
# ─── 수식 노드 DataFrame ─────────────────────────────────────────
equations_df = pd.DataFrame(EQUATIONS_META)[
    ["id", "number", "formula", "description", "page"]
]
equations_df["node_label"] = "Equation"

print("=== equations_df ===")
print(f"행 수: {len(equations_df)}")
equations_df[["id", "number", "formula", "page"]]

## Step 9. 관계(Relationship) DataFrame 생성

3가지 관계 파일을 생성합니다:
1. **IS_BELONGING_TO** — 하위 섹션 → 상위 섹션
2. **IS_NEXT_TO** — 같은 레벨의 인접 섹션
3. **REFERENCES 계열** — 섹션↔표/그림/수식

In [ ]:
def get_parent_number(number: str) -> str | None:
    """섹션 번호에서 부모 번호를 반환합니다. 예: '3.2.1' -> '3.2', '3.2' -> '3'"""
    parts = number.split(".")
    if len(parts) <= 1:
        return None  # 최상위 섹션
    return ".".join(parts[:-1])


# ─── IS_BELONGING_TO 관계 ─────────────────────────────────────────
hierarchy_rows = []
section_ids = {s["number"]: s["id"] for s in sections_data}

for sec in sections_data:
    parent_num = get_parent_number(sec["number"])
    if parent_num and parent_num in section_ids:
        hierarchy_rows.append({
            "from_id": sec["id"],                   # 자식
            "from_label": "Section",
            "relationship": "IS_BELONGING_TO",
            "to_id": section_ids[parent_num],        # 부모
            "to_label": "Section",
            "from_number": sec["number"],
            "to_number": parent_num,
        })

hierarchy_df = pd.DataFrame(hierarchy_rows)
print(f"IS_BELONGING_TO 관계 수: {len(hierarchy_df)}")
print(hierarchy_df.to_string(index=False))

In [ ]:
# ─── IS_NEXT_TO 관계 ──────────────────────────────────────────────
# 같은 레벨의 인접 섹션 쌍을 생성합니다
sequence_rows = []

# 레벨별로 그룹화
from itertools import groupby

# 부모별로 그룹화하여 같은 부모의 자식들끼리 IS_NEXT_TO 연결
def get_parent(number):
    parts = number.split(".")
    return ".".join(parts[:-1]) if len(parts) > 1 else "ROOT"

# 섹션을 부모별로 그룹화
from collections import defaultdict
parent_groups = defaultdict(list)
for sec in sections_data:
    parent = get_parent(sec["number"])
    parent_groups[parent].append(sec)

# 각 그룹 내에서 순서대로 IS_NEXT_TO 연결
for parent, children in parent_groups.items():
    # 번호 순으로 정렬
    children_sorted = sorted(children, key=lambda x: [int(p) for p in x["number"].split(".")])
    for i in range(len(children_sorted) - 1):
        curr = children_sorted[i]
        nxt  = children_sorted[i + 1]
        sequence_rows.append({
            "from_id": curr["id"],
            "from_label": "Section",
            "relationship": "IS_NEXT_TO",
            "to_id": nxt["id"],
            "to_label": "Section",
            "from_number": curr["number"],
            "to_number": nxt["number"],
        })

sequence_df = pd.DataFrame(sequence_rows)
print(f"IS_NEXT_TO 관계 수: {len(sequence_df)}")
print(sequence_df.to_string(index=False))

In [ ]:
# ─── REFERENCES / ILLUSTRATES / SUPPORTS / COMPARES / INTRODUCES ─
references_rows = []

# 그림 관계 (Section REFERENCES Figure, Figure ILLUSTRATES Section)
for fig in FIGURES_META:
    related_sec_id = f"section_{fig['related_section'].replace('.', '_')}"
    # Figure → Section: ILLUSTRATES
    references_rows.append({
        "from_id": fig["id"],
        "from_label": "Figure",
        "relationship": "ILLUSTRATES",
        "to_id": related_sec_id,
        "to_label": "Section",
        "description": fig["caption"][:80],
    })
    # Section → Figure: REFERENCES
    references_rows.append({
        "from_id": related_sec_id,
        "from_label": "Section",
        "relationship": "REFERENCES",
        "to_id": fig["id"],
        "to_label": "Figure",
        "description": f"References {fig['id']}",
    })

# 표 관계 (Table SUPPORTS Section, Section REFERENCES Table)
for tbl in TABLES_META:
    related_sec_id = f"section_{tbl['related_section'].replace('.', '_')}"
    rel_type = tbl["relationship_type"]  # SUPPORTS or COMPARES
    # Table → Section: SUPPORTS
    references_rows.append({
        "from_id": tbl["id"],
        "from_label": "Table",
        "relationship": rel_type,
        "to_id": related_sec_id,
        "to_label": "Section",
        "description": tbl["caption"][:80],
    })
    # Section → Table: REFERENCES
    references_rows.append({
        "from_id": related_sec_id,
        "from_label": "Section",
        "relationship": "REFERENCES",
        "to_id": tbl["id"],
        "to_label": "Table",
        "description": f"References {tbl['id']}",
    })

# 수식 관계 (Section INTRODUCES Equation)
for eq in EQUATIONS_META:
    related_sec_id = f"section_{eq['related_section'].replace('.', '_')}"
    references_rows.append({
        "from_id": related_sec_id,
        "from_label": "Section",
        "relationship": "INTRODUCES",
        "to_id": eq["id"],
        "to_label": "Equation",
        "description": eq["description"][:80],
    })

# Table 1은 섹션 4를 COMPARES 관계로도 연결
references_rows.append({
    "from_id": "table_1",
    "from_label": "Table",
    "relationship": "COMPARES",
    "to_id": "section_4",
    "to_label": "Section",
    "description": "Compares complexity of different layer types",
})
# Table 3은 섹션 6를 COMPARES 관계로도 연결
references_rows.append({
    "from_id": "table_3",
    "from_label": "Table",
    "relationship": "COMPARES",
    "to_id": "section_6",
    "to_label": "Section",
    "description": "Compares transformer architecture variations",
})

references_df = pd.DataFrame(references_rows)
print(f"REFERENCES 계열 관계 수: {len(references_df)}")
print("\n관계 유형별 집계:")
print(references_df["relationship"].value_counts().to_string())
references_df.head(10)

## Step 10. CSV 저장

In [ ]:
# 노드 CSV 저장
sections_df.to_csv(DATA_DIR / "nodes_sections.csv",    index=False, encoding="utf-8")
tables_df.to_csv(DATA_DIR   / "nodes_tables.csv",      index=False, encoding="utf-8")
figures_df.to_csv(DATA_DIR  / "nodes_figures.csv",     index=False, encoding="utf-8")
equations_df.to_csv(DATA_DIR / "nodes_equations.csv",  index=False, encoding="utf-8")

# 관계 CSV 저장
hierarchy_df.to_csv(DATA_DIR  / "relationships_hierarchy.csv",   index=False, encoding="utf-8")
sequence_df.to_csv(DATA_DIR   / "relationships_sequence.csv",    index=False, encoding="utf-8")
references_df.to_csv(DATA_DIR / "relationships_references.csv",  index=False, encoding="utf-8")

print("✅ CSV 파일 저장 완료")
print("\n--- 저장된 파일 목록 ---")
for f in sorted(DATA_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:45s} {size:>8,} bytes")

## Step 11. Excel 통합 파일 저장

모든 노드와 관계를 하나의 Excel 파일로 저장합니다 (여러 시트).

In [ ]:
excel_path = DATA_DIR / "transformer_knowledge_graph.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    # 노드 시트
    sections_df.to_excel(writer,    sheet_name="Nodes_Sections",   index=False)
    tables_df.to_excel(writer,      sheet_name="Nodes_Tables",     index=False)
    figures_df.to_excel(writer,     sheet_name="Nodes_Figures",    index=False)
    equations_df.to_excel(writer,   sheet_name="Nodes_Equations",  index=False)
    
    # 관계 시트
    hierarchy_df.to_excel(writer,   sheet_name="Rels_Hierarchy",   index=False)
    sequence_df.to_excel(writer,    sheet_name="Rels_Sequence",    index=False)
    references_df.to_excel(writer,  sheet_name="Rels_References",  index=False)
    
    # 요약 시트
    summary_data = {
        "Category": ["Section nodes", "Table nodes", "Figure nodes", "Equation nodes",
                     "IS_BELONGING_TO rels", "IS_NEXT_TO rels", "REFERENCES etc. rels"],
        "Count": [len(sections_df), len(tables_df), len(figures_df), len(equations_df),
                  len(hierarchy_df), len(sequence_df), len(references_df)],
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)

print(f"✅ Excel 파일 저장 완료: {excel_path}")
print(f"   파일 크기: {excel_path.stat().st_size:,} bytes")

## Step 12. 최종 검증

In [ ]:
print("=" * 60)
print("         지식 그래프 데이터 추출 결과 요약")
print("=" * 60)
print(f"\n[노드]")
print(f"  Section  노드: {len(sections_df):3d} 개")
print(f"  Table    노드: {len(tables_df):3d} 개")
print(f"  Figure   노드: {len(figures_df):3d} 개")
print(f"  Equation 노드: {len(equations_df):3d} 개")
total_nodes = len(sections_df) + len(tables_df) + len(figures_df) + len(equations_df)
print(f"  ─────────────────")
print(f"  합계           : {total_nodes:3d} 개  (+ Paper 노드 1개)")

print(f"\n[관계]")
print(f"  IS_BELONGING_TO: {len(hierarchy_df):3d} 개")
print(f"  IS_NEXT_TO     : {len(sequence_df):3d} 개")
print(f"  기타 참조 관계 : {len(references_df):3d} 개")
total_rels = len(hierarchy_df) + len(sequence_df) + len(references_df)
print(f"  ─────────────────")
print(f"  합계           : {total_rels:3d} 개  (+ PART_OF는 그래프 빌드 시 추가)")

print(f"\n[저장 파일]")
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}")

print("\n✅ 다음 단계: 02_graph_build.ipynb 실행")